# Best models per category vs. SMARD

Top picks per category, their scoreboards and two plots each, read from the exports of
[`regression-models-claude.ipynb`](regression-models-claude.ipynb) (`data/models/`) and SMARD's hourly errors (`data/metrics/`); method, leakage and caveats live there.
Pick rule: per category, the top `N_PICKS` `CANDIDATE_SPLIT` rows that beat SMARD there, on average accuracy (`overall`) or in both views of the bin (the other four).

## 1 Setup and self-check

### 1.1 Settings

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

N_PICKS = 3                 # top picks per category
MIN_BIN_SHARE = 0.5         # by-forecast hours / by-actual hours, tail categories
CANDIDATE_SPLIT = "rolling" # only these rows compete

# Label and colour per row, copied from MODELS and FIXED in regression-models-claude.ipynb
STYLE = {
    "sarimax_fourier": {"label": "SARIMAX + Fourier", "color": "#D9A53A"},
    "lgbm_direct": {"label": "LightGBM direct", "color": "#2C6EBA"},
    "lgbm_hybrid": {"label": "LightGBM hybrid", "color": "#2F8F5B"},
    "xgb_direct": {"label": "XGBoost direct", "color": "#E95D0F"},
    "xgb_hybrid": {"label": "XGBoost hybrid", "color": "#B10F0F"},
    "linear_direct": {"label": "Ridge direct", "color": "#7A4FA3"},
    "random_forest_hybrid": {"label": "Random forest hybrid", "color": "#8C564B"},
    "actual": {"label": "Actual residual load", "color": "#1C1C1C"},
    "smard": {"label": "SMARD day-ahead", "color": "#48505A", "linestyle": "--"},
}

# Bin regions and edges on a residual-load axis, never a model's colour
TAIL_COLOR = {
    "low_extreme": "#17BECF",   # cyan
    "below_zero": "#9EDAE5",    # light cyan
    "high_extreme": "#E6B800",  # gold
    "ordinary": "#E3E8EF",      # light grey
}


def label_of(row):
    """Display label of a (model, split_method) row."""
    return STYLE[row[0]]["label"]


def style_timeseries(ax, title, ylabel):
    """Apply the EDA time-series chart style to the matplotlib axes.

    `ylabel` is required: every plot must state its unit.
    """
    ax.set_title(title, loc="center", fontsize=15, pad=12)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel, color="grey")

    # Light horizontal grid lines behind the data, no box on the top and right
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    # Tick labels without tick marks
    ax.tick_params(colors="black", length=0)

    # x axis: one label per year, minor ticks every quarter
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    # y axis: thousands separator, no decimals (e.g. 45,000)
    ax.yaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")


def style_plain(ax, title, ylabel, xlabel):
    """The same look for plots without a time axis."""
    ax.set_title(title, loc="center", fontsize=15, pad=12)
    ax.set_xlabel(xlabel, color="grey")
    ax.set_ylabel(ylabel, color="grey")
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(colors="black", length=0)
    ax.yaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")

### 1.2 Loading

In [ ]:
# data/: first in the working directory, then in each parent
DATA_DIR = next(
    (folder / "data" for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(f"No data/ folder found in {Path.cwd()} or any parent folder.")

MODEL_SOURCE = "notebooks/05_modeling/regression-models-claude.ipynb with EXPORT_ENABLED = True"
SMARD_SOURCE = "notebooks/02_forecast_metrics/forecast-metrics-claude.ipynb"
SOURCES = {
    "scoreboard": ("models/model_scoreboard.csv", MODEL_SOURCE),
    "hourly": ("models/model_forecast_errors_hourly.csv", MODEL_SOURCE),
    "smard": ("metrics/smard_forecast_errors_hourly.csv", SMARD_SOURCE),
}


def load(name):
    """One export as a DataFrame; stops if the file is missing."""
    path, producer = SOURCES[name]
    path = DATA_DIR / path
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Create it by running {producer}.")
    frame = pd.read_csv(path)
    if "timestamp" in frame.columns:
        frame["timestamp"] = pd.to_datetime(frame["timestamp"], format="%Y-%m-%d %H:%M:%S")
    return frame


scoreboard = load("scoreboard")
hourly = load("hourly")
smard = load("smard").set_index("timestamp")

test_start, test_end = hourly["timestamp"].min(), hourly["timestamp"].max()
print(f"test window: {test_start:%Y-%m-%d} .. {test_end:%Y-%m-%d} ({hourly['timestamp'].nunique():,} hours)")
print("rows found :")
for split, rows in scoreboard.groupby("split_method")["model"]:
    print(f"  {split:<8}: {', '.join(rows.unique())}")
print(f"SMARD file : {smard.index.min():%Y-%m-%d %H:%M} .. {smard.index.max():%Y-%m-%d %H:%M}")

### 1.3 Rebuilt quantities

Common hours, bin edges and full months, with the rules of §7.1–§7.3 of the regression notebook.

In [ ]:
SMARD_ROW = ("smard", "none")
TEST_HOURS = pd.DatetimeIndex(hourly["timestamp"].unique()).sort_values()
TEST_DAYS = TEST_HOURS.normalize().unique()
ACTUAL = hourly.groupby("timestamp")["residual_load"].first().reindex(TEST_HOURS)

# One column per (model, split_method) row
FORECAST = hourly.pivot(index="timestamp", columns=["model", "split_method"], values="forecast")
ERROR = hourly.pivot(index="timestamp", columns=["model", "split_method"], values="err_residual_load")

# Forecast per row: SMARD first, then every row with at least one forecast
ROWS = {SMARD_ROW: smard["fc_residual_load"].reindex(TEST_HOURS)}
ROWS |= {row: FORECAST[row].reindex(TEST_HOURS) for row in FORECAST if FORECAST[row].notna().any()}

# Common hours: test hours with an actual and a forecast from every row
has_all = ACTUAL.notna() & pd.concat(ROWS, axis=1).notna().all(axis=1)
COMMON = TEST_HOURS[has_all.to_numpy()]

# Error per row on the common hours (forecast − actual)
ERRORS = {SMARD_ROW: smard.loc[COMMON, "err_residual_load"]}
ERRORS |= {row: ERROR.loc[COMMON, row] for row in ROWS if row != SMARD_ROW}

# Bin edges: quantiles of the actual over the test hours
TAIL_LEVELS = [0.01, 0.25, 0.75, 0.99]
EDGES = ACTUAL.quantile(TAIL_LEVELS)
P1, P25, P75, P99 = EDGES.to_numpy()
BINS = ["low_extreme", "ordinary", "high_extreme"]
ZERO = 0.0  # MWh: below_zero bin
REPORTED_BINS = ["low_extreme", "below_zero", "ordinary", "high_extreme"]
BIN_RULE = {
    "low_extreme": f"≤ P1 ({P1:,.0f} MWh)",
    "below_zero": "< 0 MWh",
    "ordinary": f"P25..P75 ({P25:,.0f}..{P75:,.0f} MWh)",
    "high_extreme": f"> P99 ({P99:,.0f} MWh)",
}


def tail_bin(values):
    """low_extreme (<= P1), ordinary (P25..P75), high_extreme (> P99), else other."""
    conditions = [values <= P1, (values >= P25) & (values <= P75), values > P99]
    return pd.Series(np.select(conditions, BINS, default="other"), index=values.index)


def bin_masks(values):
    """One mask per reported bin; below_zero overlaps low_extreme."""
    bins = tail_bin(values)
    masks = {name: bins == name for name in BINS}
    masks["below_zero"] = values < ZERO
    return {name: masks[name] for name in REPORTED_BINS}


# Full months: first and last day inside the test window; monthly MAE on the common hours
FULL_MONTHS = pd.PeriodIndex(
    [
        month
        for month in TEST_DAYS.to_period("M").unique()
        if month.start_time in TEST_DAYS and month.end_time.normalize() in TEST_DAYS
    ]
)
month_of = COMMON.to_period("M")
MONTHLY_MAE = pd.concat(
    {row: error.abs().groupby(month_of).mean() for row, error in ERRORS.items()}, axis=1
).loc[FULL_MONTHS]

# Scoreboard: one value and one count per row and metric
VALUE = scoreboard.pivot(index=["model", "split_method"], columns="metric", values="value")
COUNT = scoreboard.pivot(index=["model", "split_method"], columns="metric", values="count")

# Candidates: registry rows of CANDIDATE_SPLIT; SMARD and seasonal naive never compete
CANDIDATES = [
    row
    for row in VALUE.index
    if row[1] == CANDIDATE_SPLIT and row[0] not in ("smard", "seasonal_naive")
]
if not CANDIDATES:
    splits = sorted({split for _, split in VALUE.index} - {"none"})
    raise ValueError(
        f"No {CANDIDATE_SPLIT!r} rows in the exports (switched off in the last run?). "
        f"Set CANDIDATE_SPLIT to one of {splits}."
    )
missing_style = sorted({model for model, _ in CANDIDATES} - STYLE.keys())
if missing_style:
    raise KeyError(
        f"No STYLE entry for {missing_style}. "
        "Copy its label and colour from MODELS in regression-models-claude.ipynb."
    )

print(f"common hours: {len(COMMON):,} of {len(TEST_HOURS):,} test hours")
print("edges       : " + ", ".join(f"P{level:.0%}".replace("%", "") + f" {edge:,.0f}" for level, edge in EDGES.items()) + " MWh")
print(f"full months : {len(FULL_MONTHS)} ({FULL_MONTHS[0]} .. {FULL_MONTHS[-1]})")
print(f"candidates  : {len(CANDIDATES)} {CANDIDATE_SPLIT} rows")

### 1.4 Self-check

In [ ]:
REGENERATE_MODELS = f"data/models/ ({MODEL_SOURCE})"
REGENERATE_ALL = f"data/metrics/ ({SMARD_SOURCE}), then {REGENERATE_MODELS}"


def require(ok, problem, regenerate):
    """Stop the notebook, naming the files to regenerate."""
    if not ok:
        raise RuntimeError(f"self-check failed: {problem}. Regenerate {regenerate}.")


# Same actuals in the model export and the SMARD file
actual_gap = float((ACTUAL[COMMON] - smard.loc[COMMON, "residual_load"]).abs().max())
require(
    actual_gap <= 1e-6,
    f"residual_load differs between the two hourly files by up to {actual_gap:,.2f} MWh",
    REGENERATE_ALL,
)

# Same rows in the scoreboard and the hourly export
row_gap = sorted(set(VALUE.index) ^ set(ERRORS))
require(not row_gap, f"rows {row_gap} are in only one of the two model exports", REGENERATE_MODELS)

# MAE and hour count per row
for row, error in ERRORS.items():
    mae, hours = error.abs().mean(), len(error)
    exported_mae, exported_hours = VALUE.loc[row, "MAE"], COUNT.loc[row, "MAE"]
    require(
        abs(mae - exported_mae) <= 1e-6 and hours == exported_hours,
        f"{row}: MAE {mae:,.3f} on {hours:,} h, scoreboard {exported_mae:,.3f} on {exported_hours:,.0f} h",
        REGENERATE_ALL if row == SMARD_ROW else REGENERATE_MODELS,
    )

# By-actual hours per bin
for name, in_bin in bin_masks(ACTUAL[COMMON]).items():
    exported = COUNT[f"MAE_{name}_by_actual"]
    require(
        (exported == in_bin.sum()).all(),
        f"{name}: {in_bin.sum():,} by-actual hours, scoreboard {sorted(int(count) for count in exported.unique())}",
        REGENERATE_MODELS,
    )

# Months beating SMARD
for row in ERRORS:
    if row == SMARD_ROW:
        continue
    months = int((MONTHLY_MAE[row] < MONTHLY_MAE[SMARD_ROW]).sum())
    exported = (VALUE.loc[row, "months_beating_smard"], COUNT.loc[row, "months_beating_smard"])
    require(
        (months, len(FULL_MONTHS)) == exported,
        f"{row}: {months} / {len(FULL_MONTHS)} months beating SMARD, scoreboard {exported[0]:.0f} / {exported[1]:.0f}",
        REGENERATE_MODELS,
    )

print(
    f"self-check passed: exports reproduce the scoreboard "
    f"({len(ERRORS)} rows, {len(COMMON):,} common hours)"
)

## 2 Picks

Only `CANDIDATE_SPLIT` rows compete. A row qualifies in
- `overall` with a skill vs SMARD above 0; ranked by months beating SMARD, then test MAE
- a bin when its MAE is below SMARD's by actual **and** by forecast, and its by-forecast hours are at
  least `MIN_BIN_SHARE` × its by-actual hours (forecast share); ranked by MAE by actual

The first `N_PICKS` qualifying rows are picked; ties go by model key.

In [ ]:
CATEGORIES = ["overall", "ordinary", "below_zero", "low_extreme", "high_extreme"]
BIN_CATEGORIES = CATEGORIES[1:]


def forecast_share(row, name):
    """By-forecast hours / by-actual hours of a row in a bin."""
    by_actual = COUNT.loc[row, f"MAE_{name}_by_actual"]
    return COUNT.loc[row, f"MAE_{name}_by_forecast"] / by_actual if by_actual else np.nan


def failed_rules(row, category):
    """The rules a candidate fails in a category; empty if it qualifies."""
    if category == "overall":
        return [] if VALUE.loc[row, "skill_pct"] > 0 else ["skill ≤ 0"]
    failed = []
    for view in ["actual", "forecast"]:
        metric = f"MAE_{category}_by_{view}"
        if not VALUE.loc[row, metric] < VALUE.loc[SMARD_ROW, metric]:
            failed.append(f"by {view}")
    share = forecast_share(row, category)
    if not share >= MIN_BIN_SHARE:
        failed.append(f"forecast share {share:.2f} < {MIN_BIN_SHARE}")
    return failed


def rank_key(row, category):
    """Sort key: best first, ties by model key."""
    if category == "overall":
        return (-VALUE.loc[row, "months_beating_smard"], VALUE.loc[row, "MAE"], row[0])
    return (VALUE.loc[row, f"MAE_{category}_by_actual"], row[0])


RANKED_BY = {"overall": "skill > 0; months beating SMARD, then MAE"}
RANKED_BY |= {name: f"actual {BIN_RULE[name]}; MAE by actual" for name in BIN_CATEGORIES}

PICKS = {}
for category in CATEGORIES:
    qualified = sorted(
        (row for row in CANDIDATES if not failed_rules(row, category)),
        key=lambda row: rank_key(row, category),
    )
    PICKS[category] = qualified[:N_PICKS]

    print(f"{category}  ({RANKED_BY[category]})")
    if not qualified:
        print(f"  no {CANDIDATE_SPLIT} row beats SMARD in {category}")
    for rank, row in enumerate(qualified, start=1):
        status = "picked" if rank <= N_PICKS else f"not picked: qualifies, rank {rank} > N_PICKS"
        print(f"  {rank}. {label_of(row):<22} {status}")
    for row in CANDIDATES:
        if failed := failed_rules(row, category):
            print(f"  -  {label_of(row):<22} rejected: {', '.join(failed)}")
    print()

## 3 Overview

### 3.1 Picks table

One row per category and rank. For the bins, `MAE` is by the actual.

**Forecast share** = hours the row *forecasts* in the bin / hours the *actual* lies in the bin
- 1.0: the row forecasts the bin as often as it occurs; closest to 1 is best
- below 1: it forecasts the bin too rarely (0.5 = on half as many hours) and misses bin hours
- above 1: it forecasts the bin too often (no upper limit) and raises false alarms
- the pick rule only sets a floor, `MIN_BIN_SHARE`: without it, a row that rarely forecasts a tail
  value could win the by-forecast MAE on a few confident hours
- `SMARD forecast share` is the reference: SMARD's own share in the same bin
- it counts hours, not matching hours: a share of 1 can still mix misses and false alarms (flagging
  views in §6–§8)

In [ ]:
# Rows shown in 3.2 and 3.3: every pick plus SMARD, by test MAE
PICKED_IN = {}
for category, rows in PICKS.items():
    for row in rows:
        PICKED_IN.setdefault(row, []).append(category)
SHOWN = sorted([SMARD_ROW, *PICKED_IN], key=lambda row: VALUE.loc[row, "MAE"])

PICKS_COLUMNS = [
    "category",
    "rank",
    "model",
    "months beating SMARD",
    "MAE",
    "SMARD MAE",
    "skill vs SMARD (%)",
    "MAE by forecast",
    "SMARD MAE by forecast",
    "forecast share",
    "SMARD forecast share",
]
records = []
for category, rows in PICKS.items():
    for rank, row in enumerate(rows, start=1):
        record = {"category": category, "rank": rank, "model": label_of(row)}
        if category == "overall":
            months, full = VALUE.loc[row, "months_beating_smard"], COUNT.loc[row, "months_beating_smard"]
            record |= {
                "months beating SMARD": f"{months:.0f} / {full:.0f}",
                "MAE": VALUE.loc[row, "MAE"],
                "SMARD MAE": VALUE.loc[SMARD_ROW, "MAE"],
                "skill vs SMARD (%)": VALUE.loc[row, "skill_pct"],
            }
        else:
            by_actual, by_forecast = f"MAE_{category}_by_actual", f"MAE_{category}_by_forecast"
            record |= {
                "MAE": VALUE.loc[row, by_actual],
                "SMARD MAE": VALUE.loc[SMARD_ROW, by_actual],
                "skill vs SMARD (%)": VALUE.loc[row, f"skill_pct_{category}_by_actual"],
                "MAE by forecast": VALUE.loc[row, by_forecast],
                "SMARD MAE by forecast": VALUE.loc[SMARD_ROW, by_forecast],
                "forecast share": forecast_share(row, category),
                "SMARD forecast share": forecast_share(SMARD_ROW, category),
            }
        records.append(record)
PICKS_TABLE = pd.DataFrame(records, columns=PICKS_COLUMNS).set_index(["category", "rank"])

MAE_FORMAT = "{:,.0f}"
picks_formats = {
    "MAE": MAE_FORMAT,
    "SMARD MAE": MAE_FORMAT,
    "skill vs SMARD (%)": "{:+.1f}",
    "MAE by forecast": MAE_FORMAT,
    "SMARD MAE by forecast": MAE_FORMAT,
    "forecast share": "{:.2f}",
    "SMARD forecast share": "{:.2f}",
}
print(f"Picks: top {N_PICKS} {CANDIDATE_SPLIT} rows per category that beat SMARD; MAE in MWh")
if without := [category for category in CATEGORIES if not PICKS[category]]:
    print(f"  no pick in {', '.join(without)}")
display(PICKS_TABLE.style.format(picks_formats, na_rep=""))

### 3.2 Scoreboard A: accuracy

The picked rows and SMARD on all common hours, in the layout of §7.2 of the regression notebook.

In [ ]:
ACCURACY_COLUMNS = {
    "MAE": "MAE",
    "RMSE": "RMSE",
    "bias": "bias",
    "skill_pct": "skill vs SMARD (%)",
    "months_beating_smard": "months beating SMARD",
}
SCOREBOARD_A = VALUE.loc[SHOWN, list(ACCURACY_COLUMNS)].rename(columns=ACCURACY_COLUMNS)
SCOREBOARD_A.index = pd.Index([label_of(row) for row in SHOWN], name="model")
SCOREBOARD_A["picked in"] = [", ".join(PICKED_IN.get(row, [])) for row in SHOWN]

accuracy_formats = {
    "MAE": MAE_FORMAT,
    "RMSE": MAE_FORMAT,
    "bias": "{:+,.0f}",
    "skill vs SMARD (%)": "{:+.1f}",
    "months beating SMARD": "{:.0f}",
}
print(
    f"Scoreboard A: accuracy on {len(COMMON):,} common test hours "
    f"({TEST_DAYS[0]:%Y-%m-%d} .. {TEST_DAYS[-1]:%Y-%m-%d}); "
    f"MAE, RMSE and bias in MWh; {len(FULL_MONTHS)} full calendar months"
)
display(SCOREBOARD_A.style.format(accuracy_formats, na_rep="-"))

### 3.3 Scoreboard B: the bins

The same rows and order, in the layout of §7.3: binned by the actual, then by each row's own
forecast. The day maximum and minimum tables are left out.

In [ ]:
def bin_table(columns):
    """Bins as column groups, one column per (label, series) pair, rows in SHOWN order."""
    table = pd.concat(
        {(name, label): series(name) for name in REPORTED_BINS for label, series in columns.items()},
        axis=1,
    )
    table.index = pd.Index([label_of(row) for row in SHOWN], name="model")
    return table


BY_ACTUAL = bin_table(
    {
        "MAE": lambda name: VALUE.loc[SHOWN, f"MAE_{name}_by_actual"],
        "bias": lambda name: VALUE.loc[SHOWN, f"bias_{name}_by_actual"],
        "skill vs SMARD (%)": lambda name: VALUE.loc[SHOWN, f"skill_pct_{name}_by_actual"],
    }
)
BY_FORECAST = bin_table(
    {
        "MAE": lambda name: VALUE.loc[SHOWN, f"MAE_{name}_by_forecast"],
        "bias": lambda name: VALUE.loc[SHOWN, f"bias_{name}_by_forecast"],
        "hours": lambda name: COUNT.loc[SHOWN, f"MAE_{name}_by_forecast"],
        "forecast share": lambda name: pd.Series([forecast_share(row, name) for row in SHOWN], index=SHOWN),
    }
)

bin_formats = {
    "MAE": MAE_FORMAT,
    "bias": "{:+,.0f}",
    "skill vs SMARD (%)": "{:+.1f}",
    "hours": "{:,.0f}",
    "forecast share": "{:.2f}",
}
by_actual_hours = {name: int(mask.sum()) for name, mask in bin_masks(ACTUAL[COMMON]).items()}
hours_text = ", ".join(f"{name} {hours:,}" for name, hours in by_actual_hours.items())
edges_text = ", ".join(f"P{level * 100:g} = {edge:,.0f}" for level, edge in EDGES.items())
print(f"Bin edges: quantiles of the actual over the test window (MWh): {edges_text}; below_zero: actual < {ZERO:,.0f}")
for title, table in [
    (f"binned by the actual (MWh); hours: {hours_text}", BY_ACTUAL),
    ("binned by the row's own forecast (MWh); forecast share = hours / by-actual hours", BY_FORECAST),
]:
    print(f"\nScoreboard B: error in the bins, {title}")
    display(table.style.format({column: bin_formats[column[1]] for column in table.columns}, na_rep="—"))

## 4 Overall

### 4.1 Monthly skill vs SMARD

One point per full month: above 0 the pick's MAE was lower than SMARD's in that month; the legend
counts those months.

In [ ]:
def has_picks(category):
    """True if the category has picks, else print why nothing is drawn."""
    if PICKS[category]:
        return True
    print(f"no {CANDIDATE_SPLIT} row beats SMARD in {category}")
    return False


def line_style(row):
    """Colour, line style and label of a row."""
    style = STYLE[row[0]]
    return {"color": style["color"], "linestyle": style.get("linestyle", "-"), "label": label_of(row)}


def month_ticks(ax):
    """One labelled tick per month."""
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))


def legend_below(ax):
    """Legend in one row block under the axes."""
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=4, frameon=False)


if has_picks("overall"):
    fig, ax = plt.subplots(figsize=(14, 5.5))
    smard_style = line_style(SMARD_ROW) | {"label": f"{label_of(SMARD_ROW)} (0 %)"}
    ax.axhline(0, linewidth=1.6, **smard_style)
    for row in PICKS["overall"]:
        skill = 100 * (1 - MONTHLY_MAE[row] / MONTHLY_MAE[SMARD_ROW])
        months = f"{VALUE.loc[row, 'months_beating_smard']:.0f} / {len(FULL_MONTHS)}"
        style = line_style(row) | {"label": f"{label_of(row)} ({months} months)"}
        ax.plot(FULL_MONTHS.to_timestamp(), skill.to_numpy(), marker="o", markersize=6, linewidth=1.8, **style)
    style_timeseries(
        ax,
        f"Monthly skill vs SMARD, overall picks ({CANDIDATE_SPLIT}), {len(FULL_MONTHS)} full months",
        "skill vs SMARD (%)",
    )
    month_ticks(ax)
    ax.yaxis.set_major_formatter(lambda value, _: f"{value:+.0f}" if value else "0")
    legend_below(ax)
    fig.tight_layout()
    plt.show()

### 4.2 Cumulative advantage over SMARD

Running sum of `|SMARD error| − |pick error|` over the common hours: a rising line means the pick
is gaining on SMARD; a steady slope means a steady gain, steps mean a few episodes carry it.

In [ ]:
if has_picks("overall"):
    fig, ax = plt.subplots(figsize=(14, 5.5))
    ax.axhline(0, linewidth=1.6, **line_style(SMARD_ROW))
    smard_abs = ERRORS[SMARD_ROW].abs()
    for row in PICKS["overall"]:
        advantage = (smard_abs - ERRORS[row].abs()).cumsum()
        style = line_style(row) | {"label": f"{label_of(row)}: {advantage.iloc[-1]:+,.0f} MWh"}
        ax.plot(advantage.index, advantage.to_numpy(), linewidth=1.8, **style)
    style_timeseries(
        ax,
        f"Cumulative advantage over SMARD, overall picks ({CANDIDATE_SPLIT}), {len(COMMON):,} common hours",
        "cumulative |e SMARD| − |e pick| (MWh)",
    )
    month_ticks(ax)
    legend_below(ax)
    fig.tight_layout()
    plt.show()

## 5 Ordinary

### 5.1 MAE by hour of day

MAE per local hour on the ordinary hours (actual between P25 and P75): the gap between a pick's line
and SMARD's dashed line shows where in the day the pick gains or loses.

In [ ]:
BIN_HOURS = bin_masks(ACTUAL[COMMON])  # by-actual masks on the common hours


def shown_rows(category):
    """SMARD first, then the category's picks in rank order."""
    return [SMARD_ROW, *PICKS[category]]


if has_picks("ordinary"):
    in_bin = BIN_HOURS["ordinary"]
    hour = pd.Series(COMMON.hour, index=COMMON)[in_bin]
    fig, ax = plt.subplots(figsize=(14, 5.5))
    for row in shown_rows("ordinary"):
        mae = ERRORS[row][in_bin].abs().groupby(hour).mean()
        ax.plot(mae.index, mae.to_numpy(), marker="o", markersize=6, linewidth=1.8, **line_style(row))
    style_plain(
        ax,
        f"MAE by hour of day, ordinary hours (actual {BIN_RULE['ordinary']}, {in_bin.sum():,} h)",
        "MAE (MWh)",
        "hour of day (local time)",
    )
    ax.set_xticks(range(24))
    legend_below(ax)
    fig.tight_layout()
    plt.show()

### 5.2 Error range

One bar per row: on 80 % of the ordinary hours, the row's error (`forecast − actual`) lies inside
its bar; the diamond is the average error (bias). A shorter bar is a tighter forecast; a diamond
right of 0 means the row forecasts too high on average.

In [ ]:
RANGE_LEVEL = 0.8  # share of the hours inside each bar


def bin_errors(category):
    """Range of forecast − actual per row on the category's by-actual hours."""
    if not has_picks(category):
        return
    in_bin = BIN_HOURS[category]
    rows = shown_rows(category)
    tail = (1 - RANGE_LEVEL) / 2
    ranges = {row: np.quantile(ERRORS[row][in_bin], [tail, 1 - tail]) for row in rows}
    lowest = min(low for low, _ in ranges.values())
    highest = max(high for _, high in ranges.values())

    fig, ax = plt.subplots(figsize=(14, 2 + 0.9 * len(rows)))
    for position, row in enumerate(rows):
        style = line_style(row)
        color, (low, high) = style["color"], ranges[row]
        bias = ERRORS[row][in_bin].mean()
        ax.barh(
            position,
            high - low,
            left=low,
            height=0.45,
            color=f"{color}40",
            edgecolor=color,
            linestyle=style["linestyle"],
            linewidth=1.5,
        )
        ax.plot(bias, position, marker="D", markersize=9, markerfacecolor="white", markeredgecolor=color, linestyle="none")
        ax.annotate(
            f"{low:+,.0f} .. {high:+,.0f} MWh, bias {bias:+,.0f}",
            (high, position),
            xytext=(8, 0),
            textcoords="offset points",
            va="center",
            fontsize=10,
            color="0.3",
        )
    ax.axvline(0, color="0.6", linewidth=1, zorder=0)
    for text, align in [("← under-forecast ", "right"), (" over-forecast →", "left")]:
        ax.annotate(text, (0, 1), xycoords=("data", "axes fraction"), ha=align, va="bottom", fontsize=10, color="0.4",
                    xytext=(0, 4), textcoords="offset points")

    style_plain(
        ax,
        f"Error range by row, {category} hours (actual {BIN_RULE[category]}, {in_bin.sum():,} h)",
        "",
        "error (MWh), forecast − actual",
    )
    ax.set_title(ax.get_title(), fontsize=15, pad=28)  # room for the direction labels
    ax.grid(axis="y", visible=False)
    ax.grid(axis="x", color="0.9", linewidth=0.8)
    ax.xaxis.set_major_formatter(lambda value, _: f"{value:+,.0f}" if value else "0")
    ax.set_xlim(lowest - 0.1 * (highest - lowest), highest + 0.45 * (highest - lowest))
    ax.set_yticks(range(len(rows)), [label_of(row) for row in rows])
    ax.invert_yaxis()
    handles = [
        plt.Rectangle((0, 0), 1, 1, facecolor="0.85", edgecolor="0.4", label=f"middle {RANGE_LEVEL * 100:.0f} % of the hours"),
        plt.Line2D([], [], marker="D", linestyle="none", markerfacecolor="white", markeredgecolor="0.3", label="average error (bias)"),
    ]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False)
    fig.tight_layout()
    plt.show()


bin_errors("ordinary")

## 6 Below zero

### 6.1 Flagging view

One panel per row, one dot per common hour where the actual **or** the row's forecast is below
0 MWh. Tinted corner: both below (hits); below only by the actual: misses; below only by the
forecast: false alarms. Dots above the diagonal are over-forecasts. The panel title gives the
forecast share.

In [ ]:
# Bin edge, side and plain-words rule of each tail category
TAIL_EDGE = {
    "below_zero": (ZERO, "low", "below 0 MWh"),
    "low_extreme": (P1, "low", f"at or below P1 ({P1:,.0f} MWh)"),
    "high_extreme": (P99, "high", f"above P99 ({P99:,.0f} MWh)"),
}


def quiet_spot(x, y, box, span):
    """Label position inside box (x0, x1, y0, y1): few dots under it, close to the middle."""
    x0, x1, y0, y1 = box
    best, best_score = None, np.inf
    for fx in np.linspace(0.2, 0.8, 7):
        for fy in np.linspace(0.2, 0.8, 7):
            cx, cy = x0 + fx * (x1 - x0), y0 + fy * (y1 - y0)
            crowd = ((abs(x - cx) < 0.12 * span) & (abs(y - cy) < 0.06 * span)).sum()
            score = crowd + 10 * (abs(fx - 0.5) + abs(fy - 0.5))
            if score < best_score:
                best, best_score = (cx, cy), score
    return best


def tail_parity(category):
    """Forecast against actual on the hours where either lies in the bin, one panel per row."""
    if not has_picks(category):
        return
    edge, side, rule = TAIL_EDGE[category]
    actual_in = BIN_HOURS[category]
    rows = shown_rows(category)
    forecast_in = {row: bin_masks(ROWS[row][COMMON])[category] for row in rows}
    shown = {row: actual_in | forecast_in[row] for row in rows}

    # Shared, square axis range over every panel's dots
    values = pd.concat([pd.concat([ACTUAL[COMMON][shown[row]], ROWS[row][COMMON][shown[row]]]) for row in rows])
    pad = 0.05 * (values.max() - values.min())
    low, high = values.min() - pad, values.max() + pad

    # Quadrant of each count as (x0, x1, y0, y1): x is the actual, y the forecast
    inside, outside = ((low, edge), (edge, high)) if side == "low" else ((edge, high), (low, edge))
    quadrants = {"hits": (*inside, *inside), "misses": (*inside, *outside), "false alarms": (*outside, *inside)}

    fig, axes = plt.subplots(1, len(rows), figsize=(4.2 * len(rows), 5.2), sharex=True, sharey=True, layout="constrained")
    for ax, row in zip(np.atleast_1d(axes), rows):
        is_actual, is_forecast = actual_in[shown[row]], forecast_in[row][shown[row]]
        counts = {
            "hits": int((is_actual & is_forecast).sum()),
            "misses": int((is_actual & ~is_forecast).sum()),
            "false alarms": int((~is_actual & is_forecast).sum()),
        }
        require(
            counts["hits"] + counts["misses"] == COUNT.loc[row, f"MAE_{category}_by_actual"]
            and counts["hits"] + counts["false alarms"] == COUNT.loc[row, f"MAE_{category}_by_forecast"],
            f"{row} {category}: flagging counts {counts} disagree with the scoreboard hours",
            REGENERATE_MODELS,
        )

        # Tint the hits corner, draw the edge on both axes and the diagonal
        corner, size = ((low, low), edge - low) if side == "low" else ((edge, edge), high - edge)
        ax.add_patch(plt.Rectangle(corner, size, size, color=TAIL_COLOR[category], alpha=0.3, linewidth=0, zorder=0))
        ax.axvline(edge, color=TAIL_COLOR[category], linewidth=2, zorder=1)
        ax.axhline(edge, color=TAIL_COLOR[category], linewidth=2, zorder=1)
        ax.plot([low, high], [low, high], color="0.6", linewidth=1, zorder=1)
        x, y = ACTUAL[COMMON][shown[row]].to_numpy(), ROWS[row][COMMON][shown[row]].to_numpy()
        ax.scatter(x, y, s=10, color=STYLE[row[0]]["color"], alpha=0.5, linewidths=0, zorder=2)
        for name, box in quadrants.items():
            ax.annotate(
                f"{name}\n{counts[name]:,}",
                quiet_spot(x, y, box, high - low),
                ha="center",
                va="center",
                fontsize=10,
                color="0.2",
                bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8, "pad": 1.5},
                zorder=3,
            )

        share = forecast_share(row, category)
        verdict = "reference" if row == SMARD_ROW else ("passes" if share >= MIN_BIN_SHARE else "fails")
        style_plain(ax, f"{label_of(row)}\nforecast share {share:.2f} ({verdict})", "", "")
        ax.title.set_fontsize(12)
        ax.xaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")
        ax.xaxis.set_major_locator(plt.MaxNLocator(5))  # no overlapping tick labels
        ax.set_xlim(low, high)
        ax.set_ylim(low, high)
        ax.set_aspect("equal")
    np.atleast_1d(axes)[0].set_ylabel("forecast (MWh)", color="grey")
    fig.supxlabel("actual (MWh)", color="grey")
    fig.suptitle(f"Residual Load: Actual or Forecast {rule} (actual: {int(actual_in.sum()):,} h)", fontsize=15)
    plt.show()


tail_parity("below_zero")

### 6.2 Error range

One bar per row on the hours with the actual below 0 MWh: on 80 % of them the row's error lies
inside its bar; the diamond is the average error (bias).

In [ ]:
bin_errors("below_zero")

## 7 Low extreme

### 7.1 Flagging view

As in §6.1, with the edge at P1 of the actual: one dot per common hour where the actual **or** the
row's forecast is at or below P1. Tinted corner: hits; the panel title gives the forecast share.

In [ ]:
tail_parity("low_extreme")

### 7.2 Error range

One bar per row on the hours with the actual at or below P1: on 80 % of them the row's error lies
inside its bar; the diamond is the average error (bias).

In [ ]:
bin_errors("low_extreme")